# 🏗️ Notebook 1: Shopping Cart — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/shopping-cart
uv sync
```

Select the `.venv` kernel in VS Code (top-right corner of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab only needs `pydantic` — no Redis, no database. We simulate everything in plain Python so you can focus on the *ideas*.


## What we're building

An **Amazon-style shopping cart**: users browse products, click *Add to Cart*, maybe come back a week later, and eventually *Checkout*.

It sounds boring — it isn't. A cart service is one of the highest-traffic pieces of an e-commerce site, and it has three tricky problems:

1. **It has to be fast.** If *Add to Cart* takes 2 seconds, people stop shopping.
2. **It has to survive.** A user adds an item on mobile on Monday and expects it on desktop on Friday.
3. **It has to handle Prime Day.** Normal traffic × 5, with bots trying to hoard flash-sale items.

We'll design it in 3 notebooks:
1. **This one** — requirements, traffic math, high-level architecture.
2. **Next** — data model, REST APIs, idempotency, guest→user merge.
3. **Last** — runnable deep dives: inventory reservations, checkout saga, data hydration, merge algorithm.

> 💡 **Important scope note:** a shopping cart is *not* a checkout system. We only softly check that an item is in stock — we don't reserve inventory until the user actually hits **Checkout**. We'll see why in Notebook 3.

## Step 1 — Functional requirements

What the user can do:

- ➕ **Add** a product to the cart.
- 🔢 **Change quantity** or **remove** an item.
- 👀 **View** the cart with current prices and an estimated total.
- 💻 **Cross-device sync** — add on mobile, view on desktop.
- 👤 **Guest→User merge** — shop anonymously, log in later, cart items survive.
- ⚠️ **Price-change notice** — if the price of an item changed since you added it, show the new price.
- 🏷️ **Soft stock check** — tell the user "only 2 left" but don't *lock* those 2.

Out of scope for this lab: the checkout flow itself, payment processing, product search, and hard inventory reservations. Those are separate labs.

## Step 2 — Non-functional requirements

These are the constraints that *shape the architecture* more than the features do.

| Property | Target | Why |
|---|---|---|
| **Latency** | < 100 ms for add / view | Slow cart = lost sale |
| **Availability** | 99.99% | Shopping must always work |
| **Consistency** | Eventual (cross-device) | Seeing an item 1 second late on desktop is fine |
| **Durability** | Cart must survive 30 days | Users come back |
| **Scale** | Prime Day bursts | Design for the spike |

### 🔑 The big trade-off: **Availability > Strong Consistency**

In the CAP triangle, a cart picks **AP**. If a database shard is slow, we'd rather accept the write into a queue than reject the user's *Add to Cart* click. A rejected click is a direct revenue loss. A 500ms delay replicating between regions is invisible.

## Step 3 — Back-of-the-envelope (BoE)

Numbers you'd actually be asked in an interview. We'll use Amazon-scale assumptions from the reference material.

```
DAU                    = 100,000,000   # 100 M daily active users
Avg writes / user      = 5             # add / update / remove
Avg reads  / user      = 25            # view cart page
```

**Average traffic:**
- Writes:  100M × 5 = 500M / day   ≈ **5,800 writes / sec**
- Reads:   100M × 25 = 2.5B / day  ≈ **29,000 reads / sec**

**Peak (Prime Day × 5):**
- **~30,000 writes / sec**
- **~150,000 reads / sec**

**Storage:** 100M active carts × ~1 KB = **~100 GB hot data**. Small enough to fit in a Redis cluster. Persistent copy goes to a scalable KV store (DynamoDB / Cassandra).

In [ ]:
# Let's do the BoE math ourselves so the numbers aren't magic.
DAU = 100_000_000
writes_per_user_per_day = 5
reads_per_user_per_day  = 25
SECONDS_PER_DAY = 24 * 60 * 60

avg_wps = DAU * writes_per_user_per_day / SECONDS_PER_DAY
avg_rps = DAU * reads_per_user_per_day  / SECONDS_PER_DAY
peak_wps = avg_wps * 5  # Prime Day spike
peak_rps = avg_rps * 5

print(f"avg writes/sec: {avg_wps:>10,.0f}")
print(f"avg reads/sec:  {avg_rps:>10,.0f}")
print(f"peak writes/sec (Prime Day): {peak_wps:>10,.0f}")
print(f"peak reads/sec  (Prime Day): {peak_rps:>10,.0f}")

# Storage
active_carts = 100_000_000
bytes_per_cart = 1024
print(f"hot data: {active_carts * bytes_per_cart / 1e9:.0f} GB")

## Step 4 — High-level architecture

```
            ┌────────────┐
            │  Client    │  (web / mobile)
            └─────┬──────┘
                  │ HTTPS
            ┌─────▼──────┐
            │  API GW    │  auth, TLS, rate limit
            └─────┬──────┘
                  │
            ┌─────▼──────┐
            │ Cart Svc   │  stateless, auto-scales
            └──┬──┬──┬──┘
               │  │  └──────────────┐
               │  └────────┐         │
               ▼            ▼         ▼
         ┌─────────┐  ┌────────┐  ┌──────────────┐
         │ Redis   │  │ Dynamo │  │ Product Svc  │
         │ cache   │  │ (SoT)  │  │ (live prices)│
         └─────────┘  └────────┘  └──────────────┘
```

**Why each component?**

- **API Gateway** — does auth + rate limiting (e.g., 60 add-to-cart/min per user) so the Cart Svc is clean.
- **Cart Service** — stateless Python/Java/Go service. Any instance can handle any request. Scales with Kubernetes HPA.
- **Redis** — fastest possible reads. Cart lookup is a single key (`cart:{user_id}`).
- **DynamoDB (or any partitioned KV store)** — *source of truth*. Simple key-value data, sharded by `user_id`, scales writes horizontally.
- **Product Service** — owns prices & names. Cart does **not** store prices (they go stale). More on this in Notebook 3.

> 🧠 Remember: we chose NoSQL because cart access is always `get(user_id)` — no joins, no search. Relational DBs shine with joins; we don't need them here.

## Step 5 — Bad vs. Best: where do we store prices?

This is the single most common cart-design mistake. Let's look at both options side by side.

### 🚫 Bad: store the price in the cart itself

```python
cart = {
    "user_id": 1,
    "items": [
        {"sku": "A", "qty": 2, "price": 10.00, "name": "Book"}   # ← denormalized
    ]
}
```

**Why it feels right:** fast! One read returns everything.

**Why it breaks:**
1. 📉 **Stale prices.** Item was $10 last week, sale dropped it to $8 — user still sees $10.
2. 📈 Even worse: price went **up**, user sees the low price at view, the high price at checkout → angry user.
3. 🪣 **Storage bloat.** Every cart stores product metadata; duplicated across 100 M carts.

### ✅ Best: slim cart, hydrate on read

```python
cart = {"user_id": 1, "items": [{"sku": "A", "qty": 2}]}   # ← only refs
# At view time, Cart Service calls Product Service for current price.
```

Trade-off: view path does a fan-out call. We fix that with a **short-TTL product cache** (5 min) so we don't hammer the Product Service on every page refresh. Notebook 3 has a runnable demo.

This pattern — **slim source-of-truth + live hydration + tiny read cache** — shows up in Uber trip history, Netflix recommendations, LinkedIn feeds. It's worth remembering.

## Step 6 — What we deliberately left out

- **Hard inventory reservations** on *Add to Cart*. Would create DB contention and let idle carts block real buyers. We reserve only at **checkout** (Notebook 3).
- **Fancy ACID transactions across services**. We use a small **saga** instead (Notebook 3).
- **Search / recommendations.** Separate system.
- **Dedicated "save for later".** It's just a cart with a different `type` column.

Next up: the **data model and APIs** — including the surprisingly tricky `/cart/merge` endpoint.